In [ ]:
# =========================
# Cell 1 — setup, imports, paths, config
# =========================
import os
import random
import json
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import timm
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    confusion_matrix, roc_curve, auc,
    average_precision_score, precision_recall_curve,
)
from sklearn.metrics import f1_score as sk_f1

# --- Mount Drive (Colab) -------------------------------------------------
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

# --- Paths (READ-ONLY data) — identical to patched_for_multivar ----------
TRAIN_CSV     = "/content/drive/MyDrive/FYP/natural foci ground truths/data/culture_only_v02/train_dataset.csv"
VAL_CSV       = "/content/drive/MyDrive/FYP/natural foci ground truths/data/culture_only_v02/val_dataset.csv"
TEST_CSV      = "/content/drive/MyDrive/FYP/natural foci ground truths/data/culture_only_v02/test_dataset.csv"
BACKBONE_CKPT = "/content/drive/MyDrive/FYP/natural foci ground truths/backbone models/CELL_ONLY_foci_dino_backbone_DINOv03_v01.pth"

# --- Paths (WRITABLE — CLAM kept in its own subdir, nothing clobbered) ---
PAPER_REV_ROOT = "/content/drive/MyDrive/FYP/paper_revision_v03"
MODELS_DIR     = os.path.join(PAPER_REV_ROOT, "models")            # variant subfolder isolates CLAM
EVAL_DIR       = os.path.join(PAPER_REV_ROOT, "eval_results_clam") # separate from eval_results_variants
PLOTS_DIR      = os.path.join(PAPER_REV_ROOT, "training_plots")
for p in (MODELS_DIR, EVAL_DIR, PLOTS_DIR):
    os.makedirs(p, exist_ok=True)

# --- Config (matches the proposed method / other ablations) --------------
SIZE_MODE = "384"
CFG = {
    "batch_size":     256,
    "epochs":         15,
    "lr_head":        1e-3,
    "wd":             1e-4,
    "image_size":     int(SIZE_MODE),
    "backbone_name":  f"vit_tiny_patch16_{SIZE_MODE}",
    "device":         torch.device("cuda" if torch.cuda.is_available() else "cpu"),
}
DEVICE = CFG["device"]

# --- CLAM-specific hyperparameters (official CLAM defaults) --------------
CLAM_CFG = {
    "k_sample":   8,      # top/bottom-k patches used for instance clustering
    "bag_weight": 0.7,    # loss = bag_weight*bag_loss + (1-bag_weight)*inst_loss
    "att_hidden": 128,    # attention hidden dim (matches the other gated heads)
    "att_drop":   0.0,    # 0.0 = match proposed gated head exactly (controlled ablation); 0.25 = canonical CLAM
    "subtyping":  False,  # binary, mutually-exclusive -> in-class clustering on GT class
}

N_SEEDS        = 5
TARGET_RECALL  = 0.95
BOOTSTRAP_SEED = 12345
N_BOOTSTRAP    = 1000

def set_all_seeds(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

print(f"Writable root : {PAPER_REV_ROOT}")
print(f"Backbone CKPT : {BACKBONE_CKPT}")
print(f"CLAM config   : {CLAM_CFG}")
print(f"Device        : {DEVICE}")


In [ ]:
# =========================
# Cell 2 — dataset + transforms (identical to the other ablations)
# =========================
NORM_MEAN = [0.2251, 0.2251, 0.2251]
NORM_STD  = [0.2375, 0.2375, 0.2375]


def get_transforms(img_size, is_train=True):
    if is_train:
        return transforms.Compose([
            transforms.Resize((img_size, img_size),
                              interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            transforms.RandomApply([transforms.ColorJitter(brightness=0.1, contrast=0.1)], p=0.3),
            transforms.ToTensor(),
            transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
        ])
    return transforms.Compose([
        transforms.Resize((img_size, img_size),
                          interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ])


class FociDataset(Dataset):
    def __init__(self, csv_path, img_size=384, is_train=True):
        self.df = pd.read_csv(csv_path)
        self.transform = get_transforms(img_size, is_train)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(str(row["image_path"])).convert("RGB")
            return self.transform(img), torch.tensor(float(row["label"])).float()
        except Exception:
            return None


def safe_collate(batch):
    batch = [b for b in batch if b is not None]
    if not batch: return None
    xs, ys = zip(*batch)
    return torch.stack(xs), torch.stack(ys)


class FociEvalDataset(Dataset):
    def __init__(self, csv_path, img_size=384):
        self.df = pd.read_csv(csv_path)
        self.transform = get_transforms(img_size, is_train=False)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(str(row["image_path"])).convert("RGB")
            return (self.transform(img),
                    torch.tensor(float(row["label"])).float(),
                    str(row["image_path"]))
        except Exception:
            return None


def safe_collate_eval(batch):
    batch = [b for b in batch if b is not None]
    if not batch: return None
    xs, ys, paths = zip(*batch)
    return torch.stack(xs), torch.stack(ys), list(paths)


In [ ]:
# =========================
# Cell 3 — CLAM-SB model (frozen ViT-Tiny + DINO backbone -> CLAM head)
# =========================
# CLAM-SB head: gated attention bag pooling + instance-level clustering.
# Bag classifier emits a single logit (sigmoid + BCE) to stay drop-in
# compatible with the rest of the pipeline.

class CLAM_SB_Head(nn.Module):
    def __init__(self, dim, hidden_dim=128, n_classes=2,
                 k_sample=8, dropout=0.25, subtyping=False):
        super().__init__()
        self.k_sample  = k_sample
        self.n_classes = n_classes
        self.subtyping = subtyping

        self.head_norm   = nn.LayerNorm(dim)
        # gated attention (single branch)
        self.attention_V = nn.Sequential(nn.Linear(dim, hidden_dim), nn.Tanh(),    nn.Dropout(dropout))
        self.attention_U = nn.Sequential(nn.Linear(dim, hidden_dim), nn.Sigmoid(), nn.Dropout(dropout))
        self.attention_w = nn.Linear(hidden_dim, 1)

        self.bag_classifier = nn.Sequential(
            nn.Linear(dim, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 1))
        # instance classifiers — one per class (in-the-class vs out)
        self.instance_classifiers = nn.ModuleList([nn.Linear(dim, 2) for _ in range(n_classes)])
        self.inst_loss_fn = nn.CrossEntropyLoss()

    @staticmethod
    def _pos_targets(n, device): return torch.ones(n,  dtype=torch.long, device=device)
    @staticmethod
    def _neg_targets(n, device): return torch.zeros(n, dtype=torch.long, device=device)

    def _inst_eval(self, A, h, classifier):
        """In-the-class clustering: top-k attention -> positive, bottom-k -> negative."""
        device = h.device
        k = min(self.k_sample, h.size(0) // 2)
        k = max(k, 1)
        top_p_ids = torch.topk(A,  k)[1]
        top_n_ids = torch.topk(-A, k)[1]
        inst = torch.cat([h[top_p_ids], h[top_n_ids]], dim=0)
        tgt  = torch.cat([self._pos_targets(k, device), self._neg_targets(k, device)], dim=0)
        return self.inst_loss_fn(classifier(inst), tgt)

    def _inst_eval_out(self, A, h, classifier):
        """Out-of-class clustering (subtyping): top-k attention -> negative."""
        device = h.device
        k = min(self.k_sample, h.size(0))
        top_p_ids = torch.topk(A, k)[1]
        return self.inst_loss_fn(classifier(h[top_p_ids]), self._neg_targets(k, device))

    def forward(self, patches, label=None, instance_eval=False):
        # patches: (B, N, D)
        normed = self.head_norm(patches)
        A_raw  = self.attention_w(self.attention_V(normed) * self.attention_U(normed)).squeeze(-1)  # (B, N)
        A_soft = torch.softmax(A_raw, dim=1)                                # (B, N)
        M      = torch.bmm(A_soft.unsqueeze(1), patches).squeeze(1)         # (B, D) bag feature
        bag_logit = self.bag_classifier(M).squeeze(-1)                      # (B,)

        inst_loss = None
        if instance_eval and label is not None:
            losses = []
            for b in range(patches.size(0)):
                Ab, hb, lb = A_raw[b], patches[b], int(label[b].item())
                for c in range(self.n_classes):
                    clf = self.instance_classifiers[c]
                    if c == lb:                       # in-the-class clustering on GT class
                        losses.append(self._inst_eval(Ab, hb, clf))
                    elif self.subtyping:              # out-of-class only when subtyping
                        losses.append(self._inst_eval_out(Ab, hb, clf))
            if losses:
                inst_loss = torch.stack(losses).mean()
        return bag_logit, A_soft, inst_loss


class FociCLAM(nn.Module):
    def __init__(self, backbone_name, backbone_ckpt, img_size, use_imagenet=False,
                 k_sample=8, att_hidden=128, att_drop=0.25, subtyping=False):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=use_imagenet,
            num_classes=0, global_pool="", img_size=img_size,
        )
        if backbone_ckpt is not None and os.path.exists(backbone_ckpt):
            sd = torch.load(backbone_ckpt, map_location="cpu")
            self.backbone.load_state_dict(
                {k.replace("backbone.", "").replace("vit.", ""): v for k, v in sd.items()},
                strict=False,
            )
            print(f"  loaded ViT backbone weights from {backbone_ckpt}")
        elif use_imagenet:
            print(f"  loaded ImageNet-pretrained ViT: {backbone_name}")
        for p in self.backbone.parameters():
            p.requires_grad = False

        with torch.no_grad():
            feats = self.backbone.forward_features(torch.zeros(1, 3, img_size, img_size))
            if isinstance(feats, dict): feats = feats["x"]
            dim = feats.shape[-1]

        self.clam = CLAM_SB_Head(dim, hidden_dim=att_hidden, n_classes=2,
                                 k_sample=k_sample, dropout=att_drop, subtyping=subtyping)

    def _patches(self, x):
        feats = self.backbone.forward_features(x)
        if isinstance(feats, dict): feats = feats["x"]
        return feats[:, 1:, :]          # drop CLS token, keep 576 patch tokens

    def forward(self, x, label=None, instance_eval=False):
        with torch.no_grad():
            patches = self._patches(x).detach()      # backbone frozen -> treat as constant features
        return self.clam(patches, label=label, instance_eval=instance_eval)

    def trainable_head_params(self):
        return list(self.clam.parameters())


In [ ]:
# =========================
# Cell 4 — multi-seed training (combined bag + instance-clustering loss)
# =========================
VARIANTS = [
    {
        "name":          "vit_dino_clam",
        "display":       "CLAM (gated attn + instance clustering)",
        "backbone_name": CFG["backbone_name"],
        "backbone_ckpt": BACKBONE_CKPT,
        "use_imagenet":  False,
    },
]


def variant_save_path(name, seed):
    return os.path.join(MODELS_DIR, name, f"baseline_model_seed{seed}.pth")


def build_model(variant):
    return FociCLAM(
        backbone_name=variant["backbone_name"],
        backbone_ckpt=variant["backbone_ckpt"],
        img_size=CFG["image_size"],
        use_imagenet=variant["use_imagenet"],
        k_sample=CLAM_CFG["k_sample"],
        att_hidden=CLAM_CFG["att_hidden"],
        att_drop=CLAM_CFG["att_drop"],
        subtyping=CLAM_CFG["subtyping"],
    ).to(DEVICE)


def train_variant_seed(variant, seed):
    name = variant["name"]
    sp   = variant_save_path(name, seed)
    if os.path.exists(sp):
        print(f"[skip] {name} seed {seed} — checkpoint exists at {sp}")
        return {"variant": name, "seed": seed, "best_val_f1": None, "skipped": True}

    print("\n" + "#"*60)
    print(f"  Training {name} seed {seed}")
    print("#"*60)

    set_all_seeds(seed)
    os.makedirs(os.path.dirname(sp), exist_ok=True)

    train_loader = DataLoader(
        FociDataset(TRAIN_CSV, CFG["image_size"], is_train=True),
        batch_size=CFG["batch_size"], shuffle=True,
        collate_fn=safe_collate, num_workers=0,
    )
    val_loader = DataLoader(
        FociDataset(VAL_CSV, CFG["image_size"], is_train=False),
        batch_size=CFG["batch_size"], shuffle=False,
        collate_fn=safe_collate, num_workers=0,
    )

    model = build_model(variant)
    optimizer = torch.optim.AdamW(
        [{"params": model.trainable_head_params(), "lr": CFG["lr_head"]}],
        weight_decay=CFG["wd"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG["epochs"], eta_min=1e-6,
    )
    criterion  = nn.BCEWithLogitsLoss()
    bag_weight = CLAM_CFG["bag_weight"]
    best_f1 = 0.0
    hist = {"train_loss": [], "bag_loss": [], "inst_loss": [], "val_loss": [], "val_f1": []}

    for epoch in range(CFG["epochs"]):
        model.train()
        t_loss = t_bag = t_inst = 0.0; t_n = 0
        for batch in tqdm(train_loader, desc=f"{name} s{seed} ep{epoch+1}", leave=False):
            if batch is None: continue
            x, y = [b.to(DEVICE) for b in batch]
            optimizer.zero_grad()
            bag_logit, _, inst_loss = model(x, label=y, instance_eval=True)
            bag_loss = criterion(bag_logit, y)
            if inst_loss is None:
                loss = bag_loss
                inst_val = 0.0
            else:
                loss = bag_weight * bag_loss + (1.0 - bag_weight) * inst_loss
                inst_val = inst_loss.item()
            loss.backward()
            optimizer.step()
            bs = x.size(0)
            t_loss += loss.item() * bs; t_bag += bag_loss.item() * bs
            t_inst += inst_val * bs;    t_n += bs

        model.eval()
        v_loss, v_preds, v_labels = 0.0, [], []
        with torch.no_grad():
            for batch in val_loader:
                if batch is None: continue
                x, y = [b.to(DEVICE) for b in batch]
                bag_logit, _, _ = model(x)
                v_loss += criterion(bag_logit, y).item() * x.size(0)
                v_preds.extend((bag_logit >= 0.0).float().cpu().numpy())
                v_labels.extend(y.cpu().numpy())
        scheduler.step()
        val_f1 = sk_f1(v_labels, v_preds, zero_division=0)
        hist["train_loss"].append(t_loss/max(t_n, 1))
        hist["bag_loss"].append(t_bag/max(t_n, 1))
        hist["inst_loss"].append(t_inst/max(t_n, 1))
        hist["val_loss"].append(v_loss/max(len(v_labels), 1))
        hist["val_f1"].append(val_f1)
        print(f"  {name} s{seed} ep{epoch+1:02d}: loss={hist['train_loss'][-1]:.4f} "
              f"(bag={hist['bag_loss'][-1]:.4f} inst={hist['inst_loss'][-1]:.4f})  val_f1={val_f1:.4f}")
        if val_f1 >= best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), sp)

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ep = range(1, CFG["epochs"] + 1)
    ax[0].plot(ep, hist["train_loss"], marker="o", label="total")
    ax[0].plot(ep, hist["bag_loss"],   marker="o", label="bag")
    ax[0].plot(ep, hist["inst_loss"],  marker="o", label="instance")
    ax[0].plot(ep, hist["val_loss"],   marker="o", label="val")
    ax[0].set_title(f"Loss ({name} seed {seed})"); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(ep, hist["val_f1"], marker="o", color="green")
    ax[1].set_title(f"Val F1 ({name} seed {seed})"); ax[1].grid(alpha=.3)
    plt.tight_layout()
    plot_path = os.path.join(PLOTS_DIR, f"{name}_seed{seed}.png")
    plt.savefig(plot_path); plt.close()
    print(f"  done {name} s{seed}: best val F1 = {best_f1:.4f}  ->  {sp}")
    return {"variant": name, "seed": seed, "best_val_f1": float(best_f1), "skipped": False}


if __name__ == "__main__":
    print("Variants scheduled to train:")
    for v in VARIANTS:
        print(f"  - {v['name']}  ({v['display']})")
    print(f"  N_SEEDS = {N_SEEDS}")

    summary = []
    for v in VARIANTS:
        for s in range(N_SEEDS):
            summary.append(train_variant_seed(v, s))

    print("\n" + "="*60)
    print("  Training complete")
    print("="*60)
    for r in summary:
        tag = "skipped" if r["skipped"] else f"best val F1 = {r['best_val_f1']:.4f}"
        print(f"  {r['variant']:22s} seed {r['seed']}: {tag}")


In [ ]:
# =========================
# Cell 5 — multi-seed evaluation + paper_numbers + per-seed CSVs
# =========================
# Same schema as patched_for_multivar so the CLAM row drops straight into
# the MIL-aggregator ablation table (Table 9).

def infer_split(model, csv_path):
    loader = DataLoader(
        FociEvalDataset(csv_path, CFG["image_size"]),
        batch_size=CFG["batch_size"], shuffle=False,
        collate_fn=safe_collate_eval, num_workers=0,
    )
    all_probs, all_labels, all_paths = [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, leave=False):
            if batch is None: continue
            x, y, paths = batch
            x = x.to(DEVICE)
            out = model(x)
            logits = out[0] if isinstance(out, (tuple, list)) else out
            probs  = torch.sigmoid(logits)
            all_probs.extend(probs.cpu().numpy().tolist())
            all_labels.extend(y.cpu().numpy().tolist())
            all_paths.extend(paths)
    return (np.array(all_labels).astype(int),
            np.array(all_probs).astype(float),
            all_paths)


def pick_target_recall_threshold(y, p, target_recall):
    fpr, tpr, thr = roc_curve(y, p)
    valid = np.where(tpr >= target_recall)[0]
    idx = int(valid[0]) if len(valid) else int(np.argmax(tpr))
    return float(thr[idx]), float(tpr[idx]), float(fpr[idx])


def metrics_at_threshold(y, p, tau):
    yh = (p >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0, 1]).ravel()
    return {
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
        "accuracy":    float(accuracy_score(y, yh)),
        "precision":   float(precision_score(y, yh, zero_division=0)),
        "recall":      float(recall_score(y, yh, zero_division=0)),
        "f1":          float(f1_score(y, yh, zero_division=0)),
        "specificity": float(tn/(tn+fp)) if (tn+fp) else 0.0,
    }


def roc_auc_safe(y, p):
    fpr, tpr, _ = roc_curve(y, p)
    return float(auc(fpr, tpr))


def bootstrap_test_metrics(y, p, tau, n_boot, rng):
    n = len(y)
    keys = ("accuracy","precision","recall","f1","specificity","roc_auc","pr_auc")
    out  = {k: [] for k in keys}
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yt, yp = y[idx], p[idx]
        if len(np.unique(yt)) < 2: continue
        m = metrics_at_threshold(yt, yp, tau)
        try:
            r = roc_auc_safe(yt, yp)
            a = float(average_precision_score(yt, yp))
        except Exception: continue
        for k in ("accuracy","precision","recall","f1","specificity"):
            out[k].append(m[k])
        out["roc_auc"].append(r); out["pr_auc"].append(a)
    return out


def agg(vals):
    v = np.array(vals, dtype=float)
    return {"mean": float(v.mean()),
            "sd":   float(v.std(ddof=1)) if len(v) > 1 else 0.0,
            "per_seed": v.tolist()}


def boot_ci(pooled):
    if not pooled: return [float("nan"), float("nan")]
    lo, hi = np.percentile(pooled, [2.5, 97.5])
    return [float(lo), float(hi)]


print(">>> Multi-seed eval — CLAM\n")
rng = np.random.default_rng(BOOTSTRAP_SEED)
combined = {"config": {
    "n_seeds":        N_SEEDS,
    "target_recall":  TARGET_RECALL,
    "n_bootstrap":    N_BOOTSTRAP,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "clam":           CLAM_CFG,
    "val_csv":        VAL_CSV,
    "test_csv":       TEST_CSV,
}, "variants": []}

for variant in VARIANTS:
    name = variant["name"]
    print(f"\n--- {name}  ({variant['display']}) ---")
    per_seed = []
    for seed in range(N_SEEDS):
        ckpt = variant_save_path(name, seed)
        if not os.path.exists(ckpt):
            raise FileNotFoundError(f"Missing checkpoint: {ckpt}\nRun cell 4 first.")
        model = build_model(variant)
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE), strict=False)
        model.eval()

        y_v, p_v, paths_v = infer_split(model, VAL_CSV)
        y_t, p_t, paths_t = infer_split(model, TEST_CSV)

        seed_dir = os.path.join(EVAL_DIR, name, f"seed_{seed}")
        os.makedirs(seed_dir, exist_ok=True)
        pd.DataFrame({"image_path": paths_v, "label": y_v, "prob": p_v}) \
            .to_csv(os.path.join(seed_dir, "predictions_val.csv"), index=False)
        pd.DataFrame({"image_path": paths_t, "label": y_t, "prob": p_t}) \
            .to_csv(os.path.join(seed_dir, "predictions_test.csv"), index=False)

        tau_v, tpr_v, fpr_v = pick_target_recall_threshold(y_v, p_v, TARGET_RECALL)
        roc_a = roc_auc_safe(y_t, p_t)
        pr_a  = float(average_precision_score(y_t, p_t))
        m05   = metrics_at_threshold(y_t, p_t, 0.5)
        mvt   = metrics_at_threshold(y_t, p_t, tau_v)
        b05   = bootstrap_test_metrics(y_t, p_t, 0.5,   N_BOOTSTRAP, rng)
        bvt   = bootstrap_test_metrics(y_t, p_t, tau_v, N_BOOTSTRAP, rng)

        per_seed.append({
            "seed": seed, "tau_val": tau_v,
            "roc_auc": roc_a, "pr_auc": pr_a,
            "at_tau_0.5": m05, "at_tau_val": mvt,
            "boot_at_tau_0.5": b05, "boot_at_tau_val": bvt,
        })
        print(f"  seed {seed}: ROC-AUC={roc_a:.4f}  PR-AUC={pr_a:.4f}  "
              f"F1@0.5={m05['f1']:.3f}  F1@tau_val={mvt['f1']:.3f}  tau_val={tau_v:.3f}")

    def col(field):      return [r[field] for r in per_seed]
    def colop(opkey, m): return [r[opkey][m] for r in per_seed]
    def pool(opkey, m):
        out = []
        for r in per_seed:
            out.extend(r["boot_" + opkey][m])
        return out

    summary = {
        "name":     name,
        "display":  variant["display"],
        "headline": {
            "roc_auc": {**agg(col("roc_auc")), "ci_bootstrap": boot_ci(pool("at_tau_val", "roc_auc"))},
            "pr_auc":  {**agg(col("pr_auc")),  "ci_bootstrap": boot_ci(pool("at_tau_val", "pr_auc"))},
        },
        "tau_val":  {**agg(col("tau_val"))},
    }
    for opkey in ("at_tau_0.5", "at_tau_val"):
        block = {}
        for metric in ("accuracy","precision","recall","f1","specificity"):
            block[metric] = {**agg(colop(opkey, metric)),
                             "ci_bootstrap": boot_ci(pool(opkey, metric))}
        for c in ("tp","fp","fn","tn"):
            vals = colop(opkey, c)
            block[c] = {"mean": float(np.mean(vals)),
                        "sd":   float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0,
                        "per_seed": vals}
        summary[opkey] = block

    summary["per_seed"] = [{k: v for k, v in r.items() if not k.startswith("boot_")}
                           for r in per_seed]
    combined["variants"].append(summary)


json_path = os.path.join(EVAL_DIR, "paper_numbers_clam_v03.json")
with open(json_path, "w") as f:
    json.dump(combined, f, indent=2)
print(f"\nwrote {json_path}")


# --- Paste-ready markdown (matches Table 9 columns) ----------------------
def fmt(b):  return f"{b['mean']:.3f} +- {b['sd']:.3f}"

md_lines = ["# CLAM ablation results (paper_revision_v03)\n",
            f"5 seeds; bootstrap N = {N_BOOTSTRAP}. CLAM config: {CLAM_CFG}\n", "---\n",
            "## At tau = 0.5 (multi-seed, mean +- SD) — paste as the CLAM row of Table 9\n",
            "| Pooling head | F1 | Precision | Recall | ROC-AUC | PR-AUC |",
            "|---|---|---|---|---|---|"]
for v in combined["variants"]:
    op = v["at_tau_0.5"]
    md_lines.append(
        f"| {v['display']} | {fmt(op['f1'])} | {fmt(op['precision'])} "
        f"| {fmt(op['recall'])} | {fmt(v['headline']['roc_auc'])} | {fmt(v['headline']['pr_auc'])} |"
    )
md_lines += ["", "## At validation-tuned tau* (recall >= 0.95)\n",
             "| Pooling head | tau* | F1 | Precision | Recall | FN | FP |",
             "|---|---|---|---|---|---|---|"]
for v in combined["variants"]:
    op = v["at_tau_val"]
    md_lines.append(
        f"| {v['display']} | {fmt(v['tau_val'])} | {fmt(op['f1'])} "
        f"| {fmt(op['precision'])} | {fmt(op['recall'])} "
        f"| {op['fn']['mean']:.1f}+-{op['fn']['sd']:.1f} "
        f"| {op['fp']['mean']:.1f}+-{op['fp']['sd']:.1f} |"
    )

md_path = os.path.join(EVAL_DIR, "paper_numbers_clam_v03.md")
with open(md_path, "w", encoding="utf-8") as f:
    f.write("\n".join(md_lines))
print(f"wrote {md_path}")

print("\n" + "="*60)
print("  SUMMARY @ tau = 0.5")
print("="*60)
print(f"{'Pooling head':<42} {'F1':<18} {'ROC-AUC':<18} {'PR-AUC':<18}")
for v in combined["variants"]:
    print(f"{v['display']:<42} {fmt(v['at_tau_0.5']['f1']):<18} "
          f"{fmt(v['headline']['roc_auc']):<18} {fmt(v['headline']['pr_auc']):<18}")
print(f"\nArtifacts in {EVAL_DIR}")
